# Notebook 01: Exploratory Data Analysis (EDA) & Data Audit

**Role:** Role 1 (ML Lead)
**Handbook Section:** Section 2 (Role 1 Tasks, Step 1), Section 5.1/5.2, Section 7.6 (E1: EDA)

## Objectives:
- Audit the raw Kaggle Crop Recommendation dataset.
- Check dimensions (expected 2,200 rows x 8 columns).
- Inspect summary statistics (`describe()`).
- Verify zero duplicates and zero missing values.
- Confirm class balance (exactly 22 crops, 100 rows each).
- Verify min and max of each feature against physical bounds.
- Establish units and period for the rainfall column.

### Cell 1: Environment & Library Setup
Import standard data libraries (`pandas`, `numpy`) and configure display formatting for clear inspection of all 22 classes.

In [1]:
import os
import pandas as pd
import numpy as np

# Set display options for clear tabular inspection
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.precision', 2)

print(f"Pandas version: {pd.__version__}")

Pandas version: 2.3.3


### Cell 2: Load the Dataset
Locate and load `Crop_recommendation.csv` from the `data/` directory. Handles relative path resolution whether executed from project root or the `notebooks/` directory.

In [2]:
# Locate CSV path relative to current working directory
possible_paths = [
    "data/Crop_recommendation.csv",
    "../data/Crop_recommendation.csv",
    "data/crop_recommendation.csv",
    "../data/crop_recommendation.csv"
]

csv_path = next((p for p in possible_paths if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError(
        "Dataset not found! Please place 'Crop_recommendation.csv' in the 'data/' directory."
    )

df = pd.read_csv(csv_path)
print(f"Successfully loaded dataset from: {csv_path}")
df.head()

Successfully loaded dataset from: data/Crop_recommendation.csv
    N   P   K  temperature  humidity    ph  rainfall label
0  90  42  43        20.88     82.00  6.50    202.94  rice
1  85  58  41        21.77     80.32  7.04    226.66  rice
2  60  55  44        23.00     82.32  7.84    263.96  rice
3  74  35  40        26.49     80.16  6.98    242.86  rice
4  78  42  42        20.13     81.60  7.63    262.72  rice


### Cell 3: Dataset Shape and Column Names
Verify the shape matches the handbook specification: exactly 2,200 rows and 8 columns (7 features + 1 target label).

In [3]:
n_rows, n_cols = df.shape
print(f"Dataset Shape: {n_rows} rows x {n_cols} columns")
print(f"Columns: {list(df.columns)}")

# Assert required shape per project specification
assert n_rows == 2200, f"Expected 2200 rows, got {n_rows}"
assert n_cols == 8, f"Expected 8 columns, got {n_cols}"
expected_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']
assert list(df.columns) == expected_cols, f"Columns do not match expected: {expected_cols}"
print("Shape and column assertions passed.")

Dataset Shape: 2200 rows x 8 columns
Columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']
Shape and column assertions passed.


### Cell 4: Summary Statistics (`describe()`)
Compute count, mean, standard deviation, min, percentiles (25%, 50%, 75%), and max for each numerical feature.

In [4]:
stats = df.describe().T
stats['iqr'] = stats['75%'] - stats['25%']
print("Summary Statistics:")
stats

Summary Statistics:
              count    mean    std    min    25%    50%     75%     max    iqr
N            2200.0   50.55  36.92   0.00  21.00  37.00   84.25  140.00  63.25
P            2200.0   53.36  32.99   5.00  28.00  51.00   68.00  145.00  40.00
K            2200.0   48.15  50.65   5.00  20.00  32.00   49.00  205.00  29.00
temperature  2200.0   25.62   5.06   8.83  22.77  25.60   28.56   43.68   5.79
humidity     2200.0   71.48  22.26  14.26  60.26  80.47   89.95   99.98  29.69
ph           2200.0    6.47   0.77   3.50   5.97   6.43    6.92    9.94   0.95
rainfall     2200.0  103.46  54.96  20.21  64.55  94.87  124.27  298.56  59.72


### Cell 5: Duplicate Check
Check for duplicate rows. Duplicates would risk test leakage or distort cross-validation folds if not identified early.

In [5]:
n_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {n_duplicates}")
assert n_duplicates == 0, f"Found {n_duplicates} duplicate rows!"
print("Zero duplicates confirmed.")

Number of duplicate rows: 0
Zero duplicates confirmed.


### Cell 6: Missing Values Check
Verify whether any null or NaN values exist across features or labels.

In [6]:
missing_counts = df.isnull().sum()
print("Missing values per column:")
print(missing_counts)
assert missing_counts.sum() == 0, f"Missing values detected: {missing_counts.to_dict()}"
print("Zero missing values confirmed across all columns.")

Missing values per column:
N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64
Zero missing values confirmed across all columns.


### Cell 7: Class Balance Audit
Verify the distribution of crops. There must be exactly 22 distinct crops with exactly 100 rows per crop.

In [7]:
class_counts = df['label'].value_counts().sort_index()
n_classes = df['label'].nunique()

print(f"Unique crops count: {n_classes}")
print("\nDistribution across crops:")
print(class_counts)

assert n_classes == 22, f"Expected 22 classes, got {n_classes}"
assert (class_counts == 100).all(), "Class distribution is imbalanced! Expected 100 rows per class."
print("\nPerfect class balance confirmed: 22 crops x 100 rows each.")

Unique crops count: 22

Distribution across crops:
label
apple          100
banana         100
blackgram      100
chickpea       100
coconut        100
coffee         100
cotton         100
grapes         100
jute           100
kidneybeans    100
lentil         100
maize          100
mango          100
mothbeans      100
mungbean       100
muskmelon      100
orange         100
papaya         100
pigeonpeas     100
pomegranate    100
rice           100
watermelon     100
Name: count, dtype: int64

Perfect class balance confirmed: 22 crops x 100 rows each.


### Cell 8: Feature Min and Max vs Physical API Contract Bounds
Compare empirical minimum and maximum values against the physical boundaries defined in the API contract (AGENTS.md).

In [8]:
# Physical bounds defined in AGENTS.md contract
contract_bounds = {
    'N': (0, 200),
    'P': (0, 200),
    'K': (0, 300),
    'temperature': (-10.0, 60.0),
    'humidity': (0.0, 100.0),
    'ph': (0.0, 14.0),
    'rainfall': (0.0, 1000.0)
}

feature_cols = [c for c in df.columns if c != 'label']
bounds_records = []
for feat in feature_cols:
    feat_min = df[feat].min()
    feat_max = df[feat].max()
    phys_min, phys_max = contract_bounds[feat]
    in_bounds = (feat_min >= phys_min) and (feat_max <= phys_max)
    bounds_records.append({
        'feature': feat,
        'data_min': feat_min,
        'data_max': feat_max,
        'contract_min': phys_min,
        'contract_max': phys_max,
        'within_contract': in_bounds
    })

bounds_df = pd.DataFrame(bounds_records)
print("Feature Range vs Physical Contract Validation:")
bounds_df

Feature Range vs Physical Contract Validation:
       feature  data_min  data_max  contract_min  contract_max  \
0            N      0.00    140.00           0.0         200.0   
1            P      5.00    145.00           0.0         200.0   
2            K      5.00    205.00           0.0         300.0   
3  temperature      8.83     43.68         -10.0          60.0   
4     humidity     14.26     99.98           0.0         100.0   
5           ph      3.50      9.94           0.0          14.0   
6     rainfall     20.21    298.56           0.0        1000.0   

   within_contract  
0             True  
1             True  
2             True  
3             True  
4             True  
5             True  
6             True  


### Cell 9: Rainfall Unit and Coverage Assessment
Document the rainfall distribution and clarify what period the column represents (Handbook Section 2 & 8.4 requirement).

In [ ]:
print("Rainfall distribution:")
print(df['rainfall'].describe())

print("\nAgronomic Context & Time Period Analysis:")
print("- Rainfall units: millimeters (mm).")
print("- Empirical range: {:.2f} mm to {:.2f} mm (mean: {:.2f} mm).".format(
    df['rainfall'].min(), df['rainfall'].max(), df['rainfall'].mean()
))
print("- Representation: In Indian agronomic data, these values represent growing-season total rainfall (or annual average precipitation) required by the respective crops.")
print("- Critical for Role 1 Step 10 / Section 8.4: When querying weather APIs (Open-Meteo / NASA POWER) for reality checks, we must map climate precipitation totals over the growing season rather than daily or monthly values.")

Rainfall distribution:
count    2200.00
mean      103.46
std        54.96
min        20.21
25%        64.55
50%        94.87
75%       124.27
max       298.56
Name: rainfall, dtype: float64

Agronomic Context & Time Period Analysis:
- Rainfall units: millimeters (mm).
- Empirical range: 20.21 mm to 298.56 mm (mean: 103.46 mm).
- Representation: In Indian agronomic data, these values represent growing-season total rainfall (or annual average precipitation) required by the respective crops.
- Critical for Role 1 Step 10 / Section 8.4: When querying weather APIs (Open-Meteo / NASA POWER) for reality checks, we must map climate precipitation totals over the growing season rather than daily or monthly values.
